# Pipeline de dados — Olist

## Integrantes
Natan Hideaki Higuti

Breno Kenedy Sato Jacobi

Victor Hugo Allebrandt Freitag

Renato Augusto Sche

Kaique venandi Moreira 

## Pergunta de negócio

Quais categorias têm o maior valor de produtos vendidos em
pedidos entregues, por mês?

- Valor dos produtos: soma de price, sem frete.
- Mês de referência: mês da compra, em order_purchase_timestamp.
- Escopo: pedidos com order_status igual a delivered.
- Cada linha da Gold representará uma categoria em um mês.
- Métricas previstas: valor dos produtos e quantidade de
  pedidos distintos.
  

## Dataset escolhido
Olist — dados de comércio eletrônico brasileiro.

## Segundo candidato
Chinook — loja de músicas.

## Justificativa inicial
Escolhemos o Olist por permitir trabalhar com pedidos, itens e produtos,
praticando junções, tratamento de dados e análises de vendas.

In [1]:
# Confere o interpretador do kernel e a versao do pandas usada neste notebook.
# O caminho exibido deve apontar para a .venv do projeto.
import sys
import pandas as pd

print("Python utilizado:", sys.executable)
print("Versão do pandas:", pd.__version__)

Python utilizado: c:\Users\foxyg\OneDrive\Área de Trabalho\BPK 2 periodo\Pipeline de dados\Atividade aula09\pipeline-olist\.venv\Scripts\python.exe
Versão do pandas: 3.0.6


In [4]:
# Leitura exploratoria das fontes completas, ainda sem gravar a Bronze.
# O pandas reconhece tipos e campos ausentes automaticamente nesta exploracao.
URL_BASE = (
    "https://huggingface.co/datasets/"
    "bulutttt/olist-raw-data/resolve/main"
)

pedidos = pd.read_csv(f"{URL_BASE}/olist_orders_dataset.csv")
itens = pd.read_csv(f"{URL_BASE}/olist_order_items_dataset.csv")
produtos = pd.read_csv(f"{URL_BASE}/olist_products_dataset.csv")

In [5]:
# Reune as tabelas para repetir a mesma verificacao em cada fonte.
fontes = {
    "pedidos": pedidos,
    "itens": itens,
    "produtos": produtos,
}

for nome, tabela in fontes.items():
    # shape retorna a quantidade de linhas e colunas do DataFrame.
    linhas, colunas = tabela.shape
    print(f"{nome}: {linhas:,} linhas e {colunas} colunas")

pedidos: 99,441 linhas e 8 colunas
itens: 112,650 linhas e 7 colunas
produtos: 32,951 linhas e 9 colunas


In [6]:
# Exibe cinco linhas de cada fonte para reconhecer sua estrutura.
# Esta amostra inicial nao comprova a qualidade de toda a base.
for nome, tabela in fontes.items():
    print(f"\nFonte: {nome}")
    display(tabela.head())


Fonte: pedidos


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Fonte: itens


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



Fonte: produtos


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


## Avaliação do segundo candidato — Chinook

Vamos examinar as tabelas Invoice (vendas) e InvoiceLine
(itens das vendas) para comparar este dataset com o Olist.

In [7]:
from pathlib import Path
import requests
import sqlite3

# Pasta de avaliação dos candidatos, ainda fora das camadas do pipeline.
pasta_amostras = Path("amostras")
pasta_amostras.mkdir(exist_ok=True)

caminho_chinook = pasta_amostras / "chinook.sqlite"

url_chinook = (
    "https://raw.githubusercontent.com/lerocha/chinook-database/"
    "master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
)

# timeout limita a espera da requisicao; raise_for_status sinaliza erros HTTP.
resposta = requests.get(url_chinook, timeout=60)
resposta.raise_for_status()

# Salva o banco recebido no caminho relativo amostras/chinook.sqlite.
caminho_chinook.write_bytes(resposta.content)

# Consulta cinco linhas por tabela; LIMIT 5 nao mede o total de registros.
# O bloco with controla a transacao; sqlite3 nao fecha a conexao automaticamente.
# A conexao e usada somente para as consultas desta avaliacao.
with sqlite3.connect(caminho_chinook) as conexao:
    vendas_chinook = pd.read_sql_query(
        'SELECT * FROM "Invoice" LIMIT 5',
        conexao,
    )

    itens_chinook = pd.read_sql_query(
        'SELECT * FROM "InvoiceLine" LIMIT 5',
        conexao,
    )

print("Amostra de vendas:")
display(vendas_chinook)

print("Amostra de itens das vendas:")
display(itens_chinook)

Amostra de vendas:


,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2021-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,1.98
1,2,4,2021-01-02 00:00:00,Ullevålsveien 14,Oslo,NaN,Norway,0171,3.96
2,3,8,2021-01-03 00:00:00,Grétrystraat 63,Brussels,NaN,Belgium,1000,5.94
3,4,14,2021-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2021-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86


Amostra de itens das vendas:


,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1
2,3,2,6,0.99,1
3,4,2,8,0.99,1
4,5,2,10,0.99,1


## Bloco 1 — Escolha do dataset

### Comparação dos candidatos

Os dois candidatos possuem tabelas relacionadas e colunas de data
que permitem analisar vendas. Ambos foram acessados diretamente,
sem necessidade de cadastro.

No Chinook, examinamos amostras de Invoice e InvoiceLine,
relacionadas por InvoiceId. No Olist, examinamos pedidos, itens
e produtos, relacionados por order_id e product_id.

### P1. Qual dataset escolhemos e por quê?

Escolhemos o Olist porque o contexto de comércio eletrônico
brasileiro está mais alinhado ao interesse do grupo. As fontes
de pedidos, itens e produtos permitem praticar junções e analisar
vendas por categoria e período.

O Chinook também permite analisar vendas, mas preferimos
trabalhar com o contexto e as fontes do Olist.

### P2. Quantas fontes possui e quantas linhas tem a maior?

No recorte escolhido, utilizaremos três arquivos do Olist:

- Pedidos: 99.441 linhas e 8 colunas.
- Itens: 112.650 linhas e 7 colunas.
- Produtos: 32.951 linhas e 9 colunas.

A maior fonte utilizada é a de itens, com 112.650 linhas.
Esses três arquivos são parte do dataset, não sua totalidade.

### P3. Que problema de qualidade observamos na amostra?

Na tabela de produtos, identificamos uma inconsistência na
nomenclatura: product_name_lenght e product_description_lenght
utilizam "lenght" em vez de "length".

Registramos o problema sem alterar os dados nesta etapa.

### Critérios de escolha

- Três fontes que precisam ser combinadas.
- Colunas de data e categoria que permitem agregações.
- Download direto sem cadastro.

## Bloco 2 — Conhecer as fontes e definir a pergunta

Nesta etapa, examinamos as dimensões, os tipos das colunas,
as primeiras linhas e os valores ausentes de cada fonte,
sem modificar os dados originais.

In [8]:
# Diagnostico das fontes: tamanho, tipos reconhecidos e nulos por coluna.
# Este resumo apenas observa os dados; nao faz limpeza ou conversao.
for nome, tabela in fontes.items():
    print(f"\nFonte: {nome}")
    print(f"Linhas: {tabela.shape[0]:,}")
    print(f"Colunas: {tabela.shape[1]}")

    # isna marca ausencias; sum conta essas marcacoes em cada coluna.
    # As contagens nao devem ser somadas para obter linhas afetadas.
    resumo = pd.DataFrame({
        "tipo": tabela.dtypes.astype(str),
        "nulos": tabela.isna().sum(),
    })

    display(resumo)
    display(tabela.head())


Fonte: pedidos
Linhas: 99,441
Colunas: 8


,tipo,nulos
order_id,str,0
customer_id,str,0
order_status,str,0
order_purchase_timestamp,str,0
order_approved_at,str,160
order_delivered_carrier_date,str,1783
order_delivered_customer_date,str,2965
order_estimated_delivery_date,str,0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Fonte: itens
Linhas: 112,650
Colunas: 7


,tipo,nulos
order_id,str,0
order_item_id,int64,0
product_id,str,0
seller_id,str,0
shipping_limit_date,str,0
price,float64,0
freight_value,float64,0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



Fonte: produtos
Linhas: 32,951
Colunas: 9


,tipo,nulos
product_id,str,0
product_category_name,str,610
product_name_lenght,float64,610
product_description_lenght,float64,610
product_photos_qty,float64,610
product_weight_g,float64,2
product_length_cm,float64,2
product_height_cm,float64,2
product_width_cm,float64,2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [9]:
# Investiga se a ausencia da data de entrega varia conforme o status.
# assign cria uma tabela temporaria, sem modificar o DataFrame pedidos.
resumo_entregas = (
    pedidos
    .assign(
        sem_data_entrega=pedidos["order_delivered_customer_date"].isna()
    )
    .groupby("order_status")
    # size conta linhas; sum conta os valores True de sem_data_entrega.
    .agg(
        total_pedidos=("order_id", "size"),
        sem_data_entrega=("sem_data_entrega", "sum"),
    )
)

display(resumo_entregas)

,total_pedidos,sem_data_entrega
order_status,,
approved,2,2
canceled,625,619
created,5,5
delivered,96478,8
invoiced,314,314
processing,301,301
shipped,1107,1107
unavailable,609,609


### Interpretação dos nulos nas datas de entrega

A ausência de order_delivered_customer_date pode representar
uma etapa ainda não concluída do pedido, em vez de um erro.

Na base examinada, os 1.107 pedidos com status shipped estão
sem data de entrega ao cliente. Esse resultado é compatível
com pedidos enviados cuja entrega ainda não foi registrada.

Porém, também encontramos 8 pedidos com status delivered sem
data de entrega. Esses casos precisam de investigação, pois
há uma inconsistência entre o status e o preenchimento da data.

Entre os 625 pedidos cancelados, 619 não possuem data de
entrega e 6 possuem. Não podemos determinar a causa apenas
com essa contagem.

Nenhum registro foi corrigido ou removido nesta etapa.

### Relacionamentos entre as fontes

- itens.order_id se relaciona com pedidos.order_id.
- itens.product_id se relaciona com produtos.product_id.
- A data de referência será order_purchase_timestamp.

A combinação order_id + order_item_id é a chave candidata
de cada item. Sua unicidade ainda será verificada.
Também verificaremos a unicidade das chaves de pedidos e
produtos antes das junções.

### Fluxo previsto das camadas

Fonte → Bronze:
- Pedidos: order_id, order_status e order_purchase_timestamp.
- Itens: order_id, order_item_id, product_id e price.
- Produtos: product_id e product_category_name.
- Preservar os valores e acrescentar metadados de extração.

Bronze → Silver:
- Padronizar tipos e investigar duplicidades e nulos.
- Combinar itens, pedidos e produtos.
- Validar as junções e aplicar o escopo de pedidos entregues.
- Cada linha representará um item de um pedido.

Silver → Gold:
- Agregar por mês da compra e categoria.
- Calcular a soma de price e a quantidade de pedidos distintos.

### P1. Que tabela agregada responde à pergunta?
### Nessa tabela, uma linha representa o quê?

Uma tabela com mês da compra, categoria, valor dos produtos
vendidos e quantidade de pedidos distintos.

Cada linha representará uma categoria em um mês.
O valor será a soma de price, sem frete, considerando pedidos
com status delivered. Os pedidos serão contados por order_id
distinto, e não pela quantidade de linhas de itens.

Um pedido pode conter várias categorias. Portanto, as contagens
de pedidos por categoria não podem ser somadas para obter
o total geral de pedidos.

### P2. Quais fontes ou colunas não são necessárias?
### Por que deixá-las de fora é uma decisão de Extract?

Para esta pergunta, não precisamos identificar clientes,
vendedores, frete, dimensões físicas dos produtos nem suas
descrições. Também não precisamos das outras datas do pedido,
pois o mês será definido pela data da compra.

Outras fontes, como pagamentos e avaliações, não são necessárias
porque analisaremos o valor dos produtos em price, sem avaliar
pagamentos ou satisfação.

Selecionar as colunas na extração reduz o volume armazenado e
mantém as entradas alinhadas à pergunta. Essa seleção não
corrige os valores originais.

As datas de entrega foram úteis para a exploração dos nulos,
mas não são necessárias para a agregação escolhida.

### P3. Algum nulo encontrado pode carregar informação
### em vez de indicar erro?

Sim. A ausência da data de entrega pode indicar que a entrega
ainda não foi registrada, dependendo do status do pedido.

Os 1.107 pedidos com status shipped não possuem data de entrega
ao cliente. Entretanto, há também 8 pedidos delivered sem essa
data, que exigem investigação. Portanto, não interpretaremos
todos os nulos dessa coluna da mesma maneira.

## Bloco 3 — Extract: construir a Bronze

A Bronze armazenará as colunas selecionadas das fontes,
preservando seus valores e acrescentando a origem e o
horário da extração.

Nesta etapa não faremos limpeza, junções nem filtro de status.

In [10]:
# Caminhos relativos usam a pasta de execucao do notebook como referencia.
from pathlib import Path

pasta_data = Path("data")

destino_bronze = pasta_data / "bronze"
destino_silver = pasta_data / "silver"
destino_gold = pasta_data / "gold"

for pasta in [destino_bronze, destino_silver, destino_gold]:
    # parents cria os niveis necessarios; exist_ok permite repetir a celula.
    pasta.mkdir(parents=True, exist_ok=True)
    print(f"{pasta}: existe = {pasta.is_dir()}")

data\bronze: existe = True
data\silver: existe = True
data\gold: existe = True


In [12]:
from datetime import datetime, timezone

def extract(destino_bronze):
    """Le as colunas selecionadas, grava a Bronze e retorna um resumo.

    Os arquivos existentes sao substituidos nesta carga completa.
    Nao realiza limpeza, filtro de status ou juncao entre fontes.
    """
    destino_bronze = Path(destino_bronze)
    destino_bronze.mkdir(parents=True, exist_ok=True)

    # Define o arquivo e as colunas necessarias de cada fonte.
    # URL_BASE foi definido na celula de leitura exploratoria.
    fontes_extracao = {
        "pedidos": {
            "arquivo": "olist_orders_dataset.csv",
            "colunas": [
                "order_id",
                "order_status",
                "order_purchase_timestamp",
            ],
        },
        "itens": {
            "arquivo": "olist_order_items_dataset.csv",
            "colunas": [
                "order_id",
                "order_item_id",
                "product_id",
                "price",
            ],
        },
        "produtos": {
            "arquivo": "olist_products_dataset.csv",
            "colunas": [
                "product_id",
                "product_category_name",
            ],
        },
    }

    # Um horario em UTC identifica esta chamada, compartilhado pelas tres fontes.
    horario_extracao = datetime.now(timezone.utc).isoformat()
    resumo = []

    for nome, configuracao in fontes_extracao.items():
        origem = f"{URL_BASE}/{configuracao['arquivo']}"

        # Le novamente a fonte, sem depender dos DataFrames da exploracao.
        # dtype=str preserva campos como texto, incluindo datas e valores.
        # keep_default_na=False preserva vazios como ""; a Silver devera trata-los.
        try:
            tabela = pd.read_csv(
                origem,
                usecols=configuracao["colunas"],
                dtype=str,
                keep_default_na=False,
            )
        # Informa a fonte que falhou e preserva a causa original com from erro.
        except Exception as erro:
            raise RuntimeError(
                f"Falha ao ler a fonte '{nome}': {origem}"
            ) from erro

        # Conta somente as colunas selecionadas, antes dos dois metadados.
        linhas_fonte, colunas_fonte = tabela.shape

        # Acrescenta rastreabilidade sem corrigir os campos recebidos.
        tabela["_extraido_em"] = horario_extracao
        tabela["_origem"] = origem

        arquivo_destino = destino_bronze / f"{nome}.parquet"

        # O mesmo nome substitui o arquivo anterior, sem acumular linhas.
        # index=False evita salvar o indice do pandas como coluna adicional.
        try:
            tabela.to_parquet(arquivo_destino, index=False)
        except Exception as erro:
            raise RuntimeError(
                f"Falha ao gravar a Bronze de '{nome}': "
                f"{arquivo_destino}"
            ) from erro

        # O resumo usa a tabela em memoria; a verificacao em disco ocorre depois.
        resumo.append({
            "fonte": nome,
            "linhas_lidas": linhas_fonte,
            "linhas_gravadas": len(tabela),
            "colunas_fonte": colunas_fonte,
            "colunas_bronze": tabela.shape[1],
        })

    return pd.DataFrame(resumo)

In [13]:
# Executa a extracao e mostra o resumo retornado pela funcao.
# Requer executar antes as celulas de imports, URL_BASE e caminhos.
resumo_extracao = extract(destino_bronze)
display(resumo_extracao)

,fonte,linhas_lidas,linhas_gravadas,colunas_fonte,colunas_bronze
0,pedidos,99441,99441,3,5
1,itens,112650,112650,4,6
2,produtos,32951,32951,2,4


In [14]:
def contagens_bronze(pasta):
    """Le os Parquets gravados e retorna a quantidade de linhas por fonte."""
    return {
        nome: len(pd.read_parquet(pasta / f"{nome}.parquet"))
        for nome in ["pedidos", "itens", "produtos"]
    }

# Registra o estado em disco antes de executar novamente a extracao.
antes = contagens_bronze(destino_bronze)

extract(destino_bronze)

# Reabre os arquivos para verificar o resultado realmente persistido.
depois = contagens_bronze(destino_bronze)

comparacao = pd.DataFrame({
    "antes": antes,
    "depois": depois,
})

comparacao["mesma_contagem"] = (
    comparacao["antes"] == comparacao["depois"]
)

display(comparacao)

# Verifica contagens, nao igualdade de valores ou metadados.
# Se houver diferenca, assert interrompe esta celula com a mensagem abaixo.
assert antes == depois, "As contagens da Bronze mudaram."

,antes,depois,mesma_contagem
pedidos,99441,99441,True
itens,112650,112650,True
produtos,32951,32951,True


### Bloco 3 — Perguntas de verificação

#### P1. Quantas linhas e colunas tem cada arquivo da Bronze?
#### Os números batem com os da fonte original?

- Pedidos: 99.441 linhas e 5 colunas.
- Itens: 112.650 linhas e 6 colunas.
- Produtos: 32.951 linhas e 4 colunas.

As contagens de linhas coincidem com as fontes examinadas.

A Bronze contém somente as colunas selecionadas para a análise,
acrescentadas de _origem e _extraido_em. Por isso, não possui
todas as colunas dos arquivos originais.

#### P2. O que aconteceu com o número de linhas na segunda execução?

As contagens permaneceram iguais:

- Pedidos: 99.441 → 99.441.
- Itens: 112.650 → 112.650.
- Produtos: 32.951 → 32.951.

A extração substitui os mesmos arquivos Parquet em vez de
acrescentar linhas aos arquivos anteriores.

O teste confirmou estabilidade das contagens, mas não comparou
todos os valores. O horário da extração pode mudar.

#### P3. Por que a Bronze não deve receber correções?

Porque ela preserva os valores das colunas selecionadas da fonte,
permitindo rastrear sua origem e refazer as transformações quando
as regras mudarem.

Limpeza, conversão de tipos e tratamento de campos ausentes
serão realizados na Silver.

#### P4. Se a fonte sair do ar amanhã, o que continua funcionando?

Os arquivos já gravados na Bronze continuam disponíveis para
consulta e para alimentar as futuras etapas de transformação
e carga, desde que essas etapas leiam as camadas locais.

Uma nova extração dependerá da disponibilidade da fonte.
Neste momento, transform() e load() ainda não foram implementadas.